# 15. Prerequisite Relation Validation

Validate the KG agent's data-mined prerequisite relations across three dimensions:

1. **TOEIC Expert Structure** -- precision/recall vs known TOEIC part hierarchy
2. **Conditional Mastery Analysis** -- lift and chi-squared tests for prerequisite edges
3. **Predictive Utility** -- LambdaMART NDCG@10 with/without prerequisite feature

In [ ]:
import sys
import warnings
from pathlib import Path
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore")

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from agents.utils import set_global_seed, load_config
from agents.kg_agent import KnowledgeGraphAgent

SEEDS = [42, 123, 456, 789, 2024]

OUT_DIR = ROOT / "results" / "prerequisite_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

set_global_seed(42)
print(f"ROOT: {ROOT}")
print(f"Output: {OUT_DIR}")

## Load Data & Build KG

In [ ]:
from data.loader import EdNetLoader

splits_dir = ROOT / "data" / "splits"

train_df = pd.read_parquet(splits_dir / "train.parquet")
val_df = pd.read_parquet(splits_dir / "val.parquet")
test_df = pd.read_parquet(splits_dir / "test.parquet")

loader = EdNetLoader(data_dir=ROOT / "data" / "raw")
questions_df = loader.load_questions()
lectures_df = loader.load_lectures()

print(f"Train: {len(train_df):,} interactions")
print(f"Val:   {len(val_df):,} interactions")
print(f"Test:  {len(test_df):,} interactions")
print(f"Questions: {len(questions_df):,}")
print(f"Lectures:  {len(lectures_df):,}")

In [ ]:
# Build knowledge graph and mine prerequisites (train-only to prevent leakage)
kg = KnowledgeGraphAgent()
kg.build_graph(questions_df, lectures_df)
train_user_ids = set(train_df["user_id"].unique())
kg.build_prerequisites(train_df, train_user_ids=train_user_ids)

# Extract prerequisite edges
prereq_edges = [
    (u, v, d)
    for u, v, d in kg.graph.edges(data=True)
    if d.get("edge_type") == "PREREQUISITE_OF"
]
print(f"\nPrerequisite edges mined: {len(prereq_edges)}")

# Show sample edges
for u, v, d in prereq_edges[:10]:
    print(f"  {u} -> {v}  (p_fwd={d.get('p_forward', '?')}, p_bwd={d.get('p_backward', '?')})")

## Helper: Map Tags to TOEIC Parts

In [ ]:
# Build tag -> part(s) mapping from questions_df
part_col = "part_id" if "part_id" in questions_df.columns else "part"

q_df = questions_df.copy()
if q_df["tags"].dtype == object and isinstance(q_df["tags"].iloc[0], str):
    q_df["tags"] = q_df["tags"].apply(
        lambda s: [int(t) for t in s.split(";") if t.strip()]
    )

tag_to_parts = defaultdict(set)
for _, row in q_df.iterrows():
    pid = int(row[part_col])
    for tag_id in row["tags"]:
        tag_to_parts[tag_id].add(pid)

# Majority part for each tag
tag_to_majority_part = {}
for tag_id, parts in tag_to_parts.items():
    # Count which part appears most frequently for this tag
    part_counts = defaultdict(int)
    for _, row in q_df.iterrows():
        if tag_id in row["tags"]:
            part_counts[int(row[part_col])] += 1
    tag_to_majority_part[tag_id] = max(part_counts, key=part_counts.get)

print(f"Tags mapped to parts: {len(tag_to_majority_part)}")
# Show distribution
from collections import Counter
part_dist = Counter(tag_to_majority_part.values())
for p in sorted(part_dist):
    print(f"  Part {p}: {part_dist[p]} tags")

---
## Validation 1: TOEIC Expert Structure

Compare data-mined prerequisites aggregated at part level against the known TOEIC hierarchy:
- Listening (1-4) is prerequisite for Reading (5-7)
- Part 5 (Grammar) -> Part 6 (Text Completion) -> Part 7 (Comprehension)

In [ ]:
# Expert-defined prerequisite structure at part level
# Format: part_a -> [part_b, ...] means part_a is prerequisite of part_b
expert_edges = {
    5: [6, 7],
    6: [7],
    1: [5, 6, 7],
    2: [5, 6, 7],
    3: [5, 6, 7],
    4: [5, 6, 7],
}

# Flatten to set of (source, target) tuples
expert_edge_set = set()
for src, targets in expert_edges.items():
    for tgt in targets:
        expert_edge_set.add((src, tgt))

print(f"Expert prerequisite edges (part level): {len(expert_edge_set)}")
for s, t in sorted(expert_edge_set):
    print(f"  Part {s} -> Part {t}")

In [ ]:
# Aggregate data-mined tag-level prerequisites to part level
part_level_edges = defaultdict(int)  # (part_a, part_b) -> count

for u, v, d in prereq_edges:
    # u = "tag_X", v = "tag_Y"
    tag_a = int(u.replace("tag_", ""))
    tag_b = int(v.replace("tag_", ""))

    if tag_a in tag_to_majority_part and tag_b in tag_to_majority_part:
        part_a = tag_to_majority_part[tag_a]
        part_b = tag_to_majority_part[tag_b]
        if part_a != part_b:  # Only cross-part edges
            part_level_edges[(part_a, part_b)] += 1

print(f"Data-mined part-level edges: {len(part_level_edges)}")
for (pa, pb), cnt in sorted(part_level_edges.items(), key=lambda x: -x[1]):
    marker = "  [EXPERT]" if (pa, pb) in expert_edge_set else ""
    print(f"  Part {pa} -> Part {pb}: {cnt} tag edges{marker}")

In [ ]:
# Compute Precision and Recall vs expert structure
mined_edge_set = set(part_level_edges.keys())

true_positives = mined_edge_set & expert_edge_set
false_positives = mined_edge_set - expert_edge_set
false_negatives = expert_edge_set - mined_edge_set

precision = len(true_positives) / len(mined_edge_set) if mined_edge_set else 0.0
recall = len(true_positives) / len(expert_edge_set) if expert_edge_set else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print("=" * 50)
print("VALIDATION 1: TOEIC Expert Structure")
print("=" * 50)
print(f"True Positives:  {len(true_positives)} edges")
print(f"False Positives: {len(false_positives)} edges")
print(f"False Negatives: {len(false_negatives)} edges")
print(f"")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")

if true_positives:
    print(f"\nCorrectly identified:")
    for s, t in sorted(true_positives):
        print(f"  Part {s} -> Part {t}")

if false_negatives:
    print(f"\nMissed expert edges:")
    for s, t in sorted(false_negatives):
        print(f"  Part {s} -> Part {t}")

# Save results
v1_results = {
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "n_true_positives": len(true_positives),
    "n_false_positives": len(false_positives),
    "n_false_negatives": len(false_negatives),
    "true_positives": sorted([(s, t) for s, t in true_positives]),
    "false_positives": sorted([(s, t) for s, t in false_positives]),
    "false_negatives": sorted([(s, t) for s, t in false_negatives]),
}
pd.DataFrame([v1_results]).to_csv(OUT_DIR / "v1_expert_structure.csv", index=False)
print(f"\nSaved to {OUT_DIR / 'v1_expert_structure.csv'}")

In [ ]:
# Visualize: Part-level prerequisite heatmap (mined vs expert)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

parts = list(range(1, 8))

# Mined edges heatmap
mined_matrix = np.zeros((7, 7))
for (pa, pb), cnt in part_level_edges.items():
    mined_matrix[pa - 1][pb - 1] = cnt

sns.heatmap(
    mined_matrix, annot=True, fmt=".0f", cmap="Blues",
    xticklabels=parts, yticklabels=parts,
    ax=axes[0],
)
axes[0].set_title("Data-Mined Prerequisites (tag count)")
axes[0].set_xlabel("Target Part")
axes[0].set_ylabel("Source Part")

# Expert edges heatmap
expert_matrix = np.zeros((7, 7))
for s, t in expert_edge_set:
    expert_matrix[s - 1][t - 1] = 1

sns.heatmap(
    expert_matrix, annot=True, fmt=".0f", cmap="Greens",
    xticklabels=parts, yticklabels=parts,
    ax=axes[1],
)
axes[1].set_title("Expert TOEIC Structure")
axes[1].set_xlabel("Target Part")
axes[1].set_ylabel("Source Part")

plt.tight_layout()
fig.savefig(OUT_DIR / "v1_expert_heatmaps.png", bbox_inches="tight")
fig.savefig(OUT_DIR / "v1_expert_heatmaps.pdf", bbox_inches="tight")
plt.show()
print("Saved heatmaps.")

---
## Validation 2: Conditional Mastery Analysis

For each prerequisite edge A -> B:
- P(master_B | master_A) vs P(master_B | not master_A)
- Lift = P(master_B | master_A) / P(master_B)
- Chi-squared test for independence

In [ ]:
# Compute per-user, per-tag mastery from train data
def compute_user_tag_mastery(df, mastery_threshold=0.7, min_answers=5):
    """Compute mastery status for each (user, tag) pair."""
    df_copy = df.copy()
    if df_copy["tags"].dtype == object and isinstance(df_copy["tags"].iloc[0], str):
        df_copy["tags"] = df_copy["tags"].apply(
            lambda s: [int(t) for t in str(s).split(";") if t.strip()]
        )

    # Explode tags
    records = []
    for _, row in df_copy.iterrows():
        if not isinstance(row["tags"], list):
            continue
        for tag_id in row["tags"]:
            records.append({
                "user_id": row["user_id"],
                "tag_id": tag_id,
                "correct": row["correct"],
            })

    tag_df = pd.DataFrame(records)
    if tag_df.empty:
        return {}

    stats = tag_df.groupby(["user_id", "tag_id"]).agg(
        n_correct=("correct", "sum"),
        n_total=("correct", "count"),
    ).reset_index()
    stats["accuracy"] = stats["n_correct"] / stats["n_total"]
    stats["mastered"] = (stats["accuracy"] > mastery_threshold) & (stats["n_total"] >= min_answers)

    # Build {user_id: set of mastered tags}
    mastery = defaultdict(set)
    for _, row in stats[stats["mastered"]].iterrows():
        mastery[row["user_id"]].add(row["tag_id"])

    return mastery, stats


print("Computing user-tag mastery (this may take a while)...")
user_mastery, mastery_stats = compute_user_tag_mastery(train_df)
all_users = set(train_df["user_id"].unique())
print(f"Users with >= 1 mastered tag: {len(user_mastery):,} / {len(all_users):,}")
avg_mastered = np.mean([len(v) for v in user_mastery.values()]) if user_mastery else 0
print(f"Avg mastered tags per user: {avg_mastered:.1f}")

In [ ]:
def compute_edge_lift(edge_list, user_mastery, all_users):
    """Compute lift and chi-squared for a list of prerequisite edges."""
    results = []
    n_users = len(all_users)

    for u_node, v_node, data in edge_list:
        tag_a = int(u_node.replace("tag_", ""))
        tag_b = int(v_node.replace("tag_", ""))

        # Counts for contingency table
        a_and_b = 0
        a_not_b = 0
        not_a_and_b = 0
        not_a_not_b = 0

        for uid in all_users:
            mastered_tags = user_mastery.get(uid, set())
            has_a = tag_a in mastered_tags
            has_b = tag_b in mastered_tags

            if has_a and has_b:
                a_and_b += 1
            elif has_a and not has_b:
                a_not_b += 1
            elif not has_a and has_b:
                not_a_and_b += 1
            else:
                not_a_not_b += 1

        # Probabilities
        n_a = a_and_b + a_not_b
        n_not_a = not_a_and_b + not_a_not_b
        n_b = a_and_b + not_a_and_b

        p_b = n_b / n_users if n_users > 0 else 0
        p_b_given_a = a_and_b / n_a if n_a > 0 else 0
        p_b_given_not_a = not_a_and_b / n_not_a if n_not_a > 0 else 0
        lift = p_b_given_a / p_b if p_b > 0 else 0

        # Chi-squared test
        contingency = np.array([[a_and_b, a_not_b], [not_a_and_b, not_a_not_b]])
        try:
            if contingency.min() >= 0 and contingency.sum() > 0:
                chi2, p_value, dof, expected = chi2_contingency(contingency)
            else:
                chi2, p_value = 0.0, 1.0
        except ValueError:
            chi2, p_value = 0.0, 1.0

        results.append({
            "source": u_node,
            "target": v_node,
            "tag_a": tag_a,
            "tag_b": tag_b,
            "p_b": p_b,
            "p_b_given_a": p_b_given_a,
            "p_b_given_not_a": p_b_given_not_a,
            "lift": lift,
            "chi2": chi2,
            "p_value": p_value,
            "significant": p_value < 0.05,
            "n_master_a": n_a,
            "n_master_b": n_b,
        })

    return pd.DataFrame(results)


print("Computing lift for data-mined prerequisite edges...")
real_lift_df = compute_edge_lift(prereq_edges, user_mastery, all_users)
print(f"Computed for {len(real_lift_df)} edges")
print(f"\nLift statistics (real edges):")
print(real_lift_df["lift"].describe())
print(f"\nSignificant (p < 0.05): {real_lift_df['significant'].sum()} / {len(real_lift_df)}")

In [ ]:
# Generate random edges for comparison
set_global_seed(42)

tag_nodes = [
    n for n, d in kg.graph.nodes(data=True)
    if d.get("node_type") == "tag"
]

existing_edges = set((u, v) for u, v, _ in prereq_edges)

n_random = min(len(prereq_edges), 200)  # Match count or cap at 200
random_edges = []
rng = np.random.RandomState(42)

attempts = 0
while len(random_edges) < n_random and attempts < 10000:
    u = tag_nodes[rng.randint(len(tag_nodes))]
    v = tag_nodes[rng.randint(len(tag_nodes))]
    if u != v and (u, v) not in existing_edges:
        random_edges.append((u, v, {}))
        existing_edges.add((u, v))  # Avoid duplicates
    attempts += 1

print(f"Generated {len(random_edges)} random edges for comparison")

print("Computing lift for random edges...")
random_lift_df = compute_edge_lift(random_edges, user_mastery, all_users)
print(f"\nLift statistics (random edges):")
print(random_lift_df["lift"].describe())
print(f"Significant (p < 0.05): {random_lift_df['significant'].sum()} / {len(random_lift_df)}")

In [ ]:
# Summary comparison
print("=" * 60)
print("VALIDATION 2: Conditional Mastery Analysis")
print("=" * 60)
print(f"{'Metric':<35} {'Real Edges':>12} {'Random Edges':>12}")
print("-" * 60)
print(f"{'Number of edges':<35} {len(real_lift_df):>12} {len(random_lift_df):>12}")
print(f"{'Mean Lift':<35} {real_lift_df['lift'].mean():>12.3f} {random_lift_df['lift'].mean():>12.3f}")
print(f"{'Median Lift':<35} {real_lift_df['lift'].median():>12.3f} {random_lift_df['lift'].median():>12.3f}")
print(f"{'Mean P(B|A)':<35} {real_lift_df['p_b_given_a'].mean():>12.3f} {random_lift_df['p_b_given_a'].mean():>12.3f}")
print(f"{'Mean P(B|~A)':<35} {real_lift_df['p_b_given_not_a'].mean():>12.3f} {random_lift_df['p_b_given_not_a'].mean():>12.3f}")
print(f"{'Significant (p<0.05)':<35} {real_lift_df['significant'].sum():>12} {random_lift_df['significant'].sum():>12}")
print(f"{'% Significant':<35} {real_lift_df['significant'].mean()*100:>11.1f}% {random_lift_df['significant'].mean()*100:>11.1f}%")

# Save detailed results
real_lift_df.to_csv(OUT_DIR / "v2_real_edge_lifts.csv", index=False)
random_lift_df.to_csv(OUT_DIR / "v2_random_edge_lifts.csv", index=False)

In [ ]:
# Visualization: Histogram of lifts for real vs random edges
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Lift histogram
ax = axes[0]
bins = np.linspace(0, max(real_lift_df["lift"].max(), random_lift_df["lift"].max(), 3), 30)
ax.hist(real_lift_df["lift"], bins=bins, alpha=0.7, label="Data-mined", color="steelblue", edgecolor="white")
ax.hist(random_lift_df["lift"], bins=bins, alpha=0.7, label="Random", color="salmon", edgecolor="white")
ax.axvline(x=1.0, color="black", linestyle="--", linewidth=1, label="Lift = 1.0 (independent)")
ax.set_xlabel("Lift")
ax.set_ylabel("Count")
ax.set_title("Lift Distribution: Real vs Random Edges")
ax.legend()

# 2. P(B|A) vs P(B|~A)
ax = axes[1]
ax.scatter(
    real_lift_df["p_b_given_not_a"], real_lift_df["p_b_given_a"],
    alpha=0.5, label="Data-mined", s=30, color="steelblue",
)
ax.scatter(
    random_lift_df["p_b_given_not_a"], random_lift_df["p_b_given_a"],
    alpha=0.5, label="Random", s=30, color="salmon",
)
lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, lim], [0, lim], "k--", linewidth=1, label="y = x")
ax.set_xlabel(r"P(master$_B$ | $\neg$master$_A$)")
ax.set_ylabel(r"P(master$_B$ | master$_A$)")
ax.set_title("Conditional Mastery Probabilities")
ax.legend()

# 3. Chi-squared p-value distribution
ax = axes[2]
ax.hist(
    real_lift_df["p_value"], bins=20, alpha=0.7,
    label="Data-mined", color="steelblue", edgecolor="white",
)
ax.hist(
    random_lift_df["p_value"], bins=20, alpha=0.7,
    label="Random", color="salmon", edgecolor="white",
)
ax.axvline(x=0.05, color="red", linestyle="--", linewidth=1, label="p = 0.05")
ax.set_xlabel("p-value (chi-squared)")
ax.set_ylabel("Count")
ax.set_title(r"$\chi^2$ Test p-value Distribution")
ax.legend()

plt.tight_layout()
fig.savefig(OUT_DIR / "v2_mastery_analysis.png", bbox_inches="tight")
fig.savefig(OUT_DIR / "v2_mastery_analysis.pdf", bbox_inches="tight")
plt.show()
print("Saved mastery analysis figures.")

---
## Validation 3: Predictive Utility

Train LambdaMART with and without the `prerequisite_completion` feature to measure its impact on NDCG@10.

In [ ]:
# Build prerequisite_completion feature for each (user, question) pair
def compute_prereq_completion(df, kg, user_mastery):
    """
    For each (user, question), compute the fraction of prerequisite
    tags that the user has mastered.

    If question covers tag_B and tag_A -> tag_B is a prerequisite edge,
    then prereq_completion = fraction of such tag_A that user has mastered.
    """
    # Build tag -> list of prerequisite tags
    prereq_map = defaultdict(set)  # tag_B -> {tag_A, ...}
    for u, v, d in kg.graph.edges(data=True):
        if d.get("edge_type") == "PREREQUISITE_OF":
            tag_a = int(u.replace("tag_", ""))
            tag_b = int(v.replace("tag_", ""))
            prereq_map[tag_b].add(tag_a)

    # Build question -> tags mapping
    df_copy = df.copy()
    if df_copy["tags"].dtype == object and isinstance(df_copy["tags"].iloc[0], str):
        df_copy["tags"] = df_copy["tags"].apply(
            lambda s: [int(t) for t in str(s).split(";") if t.strip()]
        )

    prereq_scores = []
    for _, row in df_copy.iterrows():
        uid = row["user_id"]
        tags = row["tags"] if isinstance(row["tags"], list) else []
        user_mastered = user_mastery.get(uid, set())

        # Collect all prerequisite tags for this question's tags
        all_prereqs = set()
        for tag_id in tags:
            all_prereqs.update(prereq_map.get(tag_id, set()))

        if all_prereqs:
            score = len(all_prereqs & user_mastered) / len(all_prereqs)
        else:
            score = 1.0  # No prerequisites -> fully completed

        prereq_scores.append(score)

    return prereq_scores


print("Computing prerequisite completion features...")
train_prereq = compute_prereq_completion(train_df, kg, user_mastery)

# Also compute mastery for val/test users
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
user_mastery_all, _ = compute_user_tag_mastery(all_df)

val_prereq = compute_prereq_completion(val_df, kg, user_mastery_all)
test_prereq = compute_prereq_completion(test_df, kg, user_mastery_all)

print(f"Train prereq feature stats: mean={np.mean(train_prereq):.3f}, std={np.std(train_prereq):.3f}")
print(f"Val   prereq feature stats: mean={np.mean(val_prereq):.3f}, std={np.std(val_prereq):.3f}")
print(f"Test  prereq feature stats: mean={np.mean(test_prereq):.3f}, std={np.std(test_prereq):.3f}")

In [ ]:
# Prepare LambdaMART features
import lightgbm as lgb

def prepare_ltr_features(df, prereq_scores, include_prereq=True):
    """Prepare features for LambdaMART learning-to-rank."""
    feature_cols = []
    feat_df = pd.DataFrame()

    # Basic features available in the interactions
    if "elapsed_time" in df.columns:
        feat_df["elapsed_time"] = df["elapsed_time"].fillna(0).values
        feature_cols.append("elapsed_time")

    if "response_count" in df.columns:
        feat_df["response_count"] = df["response_count"].fillna(1).values
        feature_cols.append("response_count")

    # Part as feature
    part_col = "part_id" if "part_id" in df.columns else "part"
    if part_col in df.columns:
        feat_df["part_id"] = df[part_col].fillna(0).astype(int).values
        feature_cols.append("part_id")

    # Tag count
    if "tags" in df.columns:
        def tag_count(x):
            if isinstance(x, list):
                return len(x)
            if isinstance(x, str):
                return len([t for t in x.split(";") if t.strip()])
            return 0
        feat_df["n_tags"] = df["tags"].apply(tag_count).values
        feature_cols.append("n_tags")

    # Prerequisite completion feature
    if include_prereq:
        feat_df["prereq_completion"] = prereq_scores
        feature_cols.append("prereq_completion")

    labels = df["correct"].astype(int).values
    groups = df.groupby("user_id").size().values

    return feat_df[feature_cols].values, labels, groups, feature_cols


def ndcg_at_k(y_true, y_pred, groups, k=10):
    """Compute NDCG@k averaged over query groups."""
    ndcgs = []
    idx = 0
    for g in groups:
        g_true = y_true[idx:idx + g]
        g_pred = y_pred[idx:idx + g]
        idx += g

        if len(g_true) == 0 or g_true.sum() == 0:
            continue

        # Sort by predicted scores
        order = np.argsort(-g_pred)
        g_true_sorted = g_true[order][:k]

        # DCG
        discounts = np.log2(np.arange(len(g_true_sorted)) + 2)
        dcg = (g_true_sorted / discounts).sum()

        # Ideal DCG
        ideal_order = np.argsort(-g_true)
        ideal_sorted = g_true[ideal_order][:k]
        ideal_discounts = np.log2(np.arange(len(ideal_sorted)) + 2)
        idcg = (ideal_sorted / ideal_discounts).sum()

        if idcg > 0:
            ndcgs.append(dcg / idcg)

    return np.mean(ndcgs) if ndcgs else 0.0


print("Feature preparation functions defined.")

In [ ]:
# Multi-seed LambdaMART experiment: with vs without prerequisite feature
results_with = []
results_without = []

# Sort data by user_id to ensure consistent grouping
train_sorted = train_df.sort_values("user_id").reset_index(drop=True)
val_sorted = val_df.sort_values("user_id").reset_index(drop=True)
test_sorted = test_df.sort_values("user_id").reset_index(drop=True)

# Recompute prereq features for sorted data
train_prereq_sorted = compute_prereq_completion(train_sorted, kg, user_mastery)
val_prereq_sorted = compute_prereq_completion(val_sorted, kg, user_mastery_all)
test_prereq_sorted = compute_prereq_completion(test_sorted, kg, user_mastery_all)

for seed in SEEDS:
    set_global_seed(seed)
    print(f"\n--- Seed {seed} ---")

    # WITH prerequisite feature
    X_train, y_train, g_train, feat_names = prepare_ltr_features(
        train_sorted, train_prereq_sorted, include_prereq=True
    )
    X_val, y_val, g_val, _ = prepare_ltr_features(
        val_sorted, val_prereq_sorted, include_prereq=True
    )
    X_test, y_test, g_test, _ = prepare_ltr_features(
        test_sorted, test_prereq_sorted, include_prereq=True
    )

    train_set = lgb.Dataset(X_train, label=y_train, group=g_train)
    val_set = lgb.Dataset(X_val, label=y_val, group=g_val, reference=train_set)

    params_with = {
        "objective": "lambdarank",
        "metric": "ndcg",
        "eval_at": [10],
        "num_leaves": 31,
        "learning_rate": 0.05,
        "min_data_in_leaf": 20,
        "seed": seed,
        "verbose": -1,
    }

    model_with = lgb.train(
        params_with,
        train_set,
        num_boost_round=300,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)],
    )

    preds_with = model_with.predict(X_test)
    ndcg_with = ndcg_at_k(y_test, preds_with, g_test, k=10)
    results_with.append(ndcg_with)
    print(f"  WITH prereq feature:    NDCG@10 = {ndcg_with:.4f}  (features: {feat_names})")

    # WITHOUT prerequisite feature
    X_train_no, y_train_no, g_train_no, feat_names_no = prepare_ltr_features(
        train_sorted, train_prereq_sorted, include_prereq=False
    )
    X_val_no, y_val_no, g_val_no, _ = prepare_ltr_features(
        val_sorted, val_prereq_sorted, include_prereq=False
    )
    X_test_no, y_test_no, g_test_no, _ = prepare_ltr_features(
        test_sorted, test_prereq_sorted, include_prereq=False
    )

    train_set_no = lgb.Dataset(X_train_no, label=y_train_no, group=g_train_no)
    val_set_no = lgb.Dataset(X_val_no, label=y_val_no, group=g_val_no, reference=train_set_no)

    params_without = params_with.copy()

    model_without = lgb.train(
        params_without,
        train_set_no,
        num_boost_round=300,
        valid_sets=[val_set_no],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)],
    )

    preds_without = model_without.predict(X_test_no)
    ndcg_without = ndcg_at_k(y_test_no, preds_without, g_test_no, k=10)
    results_without.append(ndcg_without)
    print(f"  WITHOUT prereq feature: NDCG@10 = {ndcg_without:.4f}  (features: {feat_names_no})")
    print(f"  Delta NDCG@10 = {ndcg_with - ndcg_without:+.4f}")

In [ ]:
# Summary of predictive utility
results_with = np.array(results_with)
results_without = np.array(results_without)
deltas = results_with - results_without

print("=" * 60)
print("VALIDATION 3: Predictive Utility (LambdaMART)")
print("=" * 60)
print(f"{'Seed':<10} {'With Prereq':>12} {'Without':>12} {'Delta':>12}")
print("-" * 50)
for i, seed in enumerate(SEEDS):
    print(f"{seed:<10} {results_with[i]:>12.4f} {results_without[i]:>12.4f} {deltas[i]:>+12.4f}")
print("-" * 50)
print(f"{'Mean':<10} {results_with.mean():>12.4f} {results_without.mean():>12.4f} {deltas.mean():>+12.4f}")
print(f"{'Std':<10} {results_with.std():>12.4f} {results_without.std():>12.4f} {deltas.std():>12.4f}")

# Paired t-test
from scipy.stats import ttest_rel
t_stat, p_val = ttest_rel(results_with, results_without)
print(f"\nPaired t-test: t={t_stat:.4f}, p={p_val:.4f}")
print(f"Significant improvement (p < 0.05): {p_val < 0.05}")

# Save
v3_df = pd.DataFrame({
    "seed": SEEDS,
    "ndcg10_with_prereq": results_with,
    "ndcg10_without_prereq": results_without,
    "delta_ndcg10": deltas,
})
v3_df.to_csv(OUT_DIR / "v3_predictive_utility.csv", index=False)
print(f"\nSaved to {OUT_DIR / 'v3_predictive_utility.csv'}")

In [ ]:
# Visualization: Predictive utility comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart: with vs without, per seed
ax = axes[0]
x = np.arange(len(SEEDS))
width = 0.35
bars1 = ax.bar(x - width / 2, results_with, width, label="With prereq", color="steelblue", edgecolor="white")
bars2 = ax.bar(x + width / 2, results_without, width, label="Without prereq", color="salmon", edgecolor="white")
ax.set_xlabel("Seed")
ax.set_ylabel("NDCG@10")
ax.set_title("LambdaMART NDCG@10: With vs Without Prerequisite Feature")
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in SEEDS])
ax.legend()
# Set y-axis to start from a reasonable value for visibility
y_min = min(results_with.min(), results_without.min()) * 0.95
y_max = max(results_with.max(), results_without.max()) * 1.05
ax.set_ylim(y_min, y_max)

# Delta chart
ax = axes[1]
colors = ["green" if d > 0 else "red" for d in deltas]
ax.bar(x, deltas, color=colors, edgecolor="white", alpha=0.8)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.axhline(y=deltas.mean(), color="blue", linestyle="--", linewidth=1, label=f"Mean delta = {deltas.mean():+.4f}")
ax.set_xlabel("Seed")
ax.set_ylabel(r"$\Delta$ NDCG@10")
ax.set_title(r"$\Delta$ NDCG@10 (With - Without Prerequisite Feature)")
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in SEEDS])
ax.legend()

plt.tight_layout()
fig.savefig(OUT_DIR / "v3_predictive_utility.png", bbox_inches="tight")
fig.savefig(OUT_DIR / "v3_predictive_utility.pdf", bbox_inches="tight")
plt.show()
print("Saved predictive utility figures.")

---
## Summary

In [ ]:
print("=" * 70)
print("PREREQUISITE VALIDATION SUMMARY")
print("=" * 70)

print(f"\n1. TOEIC Expert Structure:")
print(f"   Precision = {precision:.3f}, Recall = {recall:.3f}, F1 = {f1:.3f}")
print(f"   {len(true_positives)}/{len(expert_edge_set)} expert edges recovered")

print(f"\n2. Conditional Mastery Analysis:")
print(f"   Real edges:   mean lift = {real_lift_df['lift'].mean():.3f}, "
      f"{real_lift_df['significant'].sum()}/{len(real_lift_df)} significant")
print(f"   Random edges: mean lift = {random_lift_df['lift'].mean():.3f}, "
      f"{random_lift_df['significant'].sum()}/{len(random_lift_df)} significant")

print(f"\n3. Predictive Utility (LambdaMART):")
print(f"   NDCG@10 with prereq:    {results_with.mean():.4f} +/- {results_with.std():.4f}")
print(f"   NDCG@10 without prereq: {results_without.mean():.4f} +/- {results_without.std():.4f}")
print(f"   Delta NDCG@10:          {deltas.mean():+.4f} +/- {deltas.std():.4f}")
print(f"   Paired t-test: t={t_stat:.4f}, p={p_val:.4f}")

print(f"\nAll results saved to: {OUT_DIR}")
print("=" * 70)